# Exp0.2.2 — Capacity-preserving anti-persistent-firing loss

Analysis-only notebook. Training, checkpoint evaluation, and per-run 2x3 diagnostic figures are produced by the Slurm pipeline. This notebook only reads finalized artifacts.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path('..').resolve()
ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_0_2_2_capacity_preserving_loss' / 'capacity_preserving_loss_v1'
summary = pd.read_csv(ROOT / 'summary.csv')
comparison = pd.read_csv(ROOT / 'comparison_summary.csv')
calibration = pd.read_csv(ROOT / 'calibration_summary.csv')
history = pd.read_csv(ROOT / 'history_long.csv')
shift_history = pd.read_csv(ROOT / 'shift_history_long.csv')
display(summary.sort_values(['architecture', 'test_balanced_accuracy'], ascending=[True, False]))

## Final comparison
Primary classification endpoint is validation-selected test balanced accuracy. The firing endpoint is whether long valid FR is preserved while long tail/valid ratio falls.

In [ ]:
cols = [c for c in [
    'architecture', 'condition',
    'best_val_balanced_accuracy_mean', 'test_balanced_accuracy_mean',
    'test_long_valid_fr_hz_mean', 'test_long_tail_fr_hz_mean',
    'test_long_tail_valid_ratio_mean',
    'output_weight_energy_long_fraction_mean',
    'output_weight_norm_s6_mean', 'output_weight_norm_s7_mean',
] if c in comparison.columns]
display(comparison[cols].sort_values(['architecture', 'test_balanced_accuracy_mean'], ascending=[True, False]))

## Mean train/validation balanced accuracy across seeds

In [ ]:
for architecture in sorted(history['architecture'].unique()):
    frame = history[history['architecture'] == architecture]
    fig, ax = plt.subplots(figsize=(9, 5))
    for condition, group in frame.groupby('condition'):
        curve = group.groupby('epoch', as_index=False)['val_balanced_accuracy'].mean()
        ax.plot(curve['epoch'], curve['val_balanced_accuracy'], label=f'{condition} val')
    ax.set_ylim(0, 1)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation balanced accuracy')
    ax.set_title(architecture)
    ax.axvline(5, linestyle=':')
    ax.axvline(15, linestyle=':')
    ax.legend(fontsize=8)
    plt.show()

## Loss decomposition
The key optimization view overlays task loss, weighted regularizer contribution, and total loss.

In [ ]:
ARCHITECTURE = 'short_mid_long'
CONDITION = 'sat_relative_tail_capacity'
SEED = 11
frame = history[(history.architecture == ARCHITECTURE) & (history.condition == CONDITION) & (history.seed == SEED)].sort_values('epoch')
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(frame['epoch'], frame['train_optim_task_loss'], label='task')
ax.plot(frame['epoch'], frame['train_optim_weighted_reg_loss'], label='weighted regularizer')
ax.plot(frame['epoch'], frame['train_optim_total_loss'], label='total')
ax.set_xlabel('Epoch')
ax.set_ylabel('Optimization loss')
ax.legend()
plt.show()

## Long-timescale firing dynamics and output utilization

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(frame['epoch'], frame['val_long_valid_fr_hz'], label='val long valid FR')
ax.plot(frame['epoch'], frame['val_long_tail_fr_hz'], label='val long tail FR')
ax.set_xlabel('Epoch')
ax.set_ylabel('Hz / neuron')
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(frame['epoch'], frame['output_weight_norm_s6'], label='s6 outgoing norm')
ax.plot(frame['epoch'], frame['output_weight_norm_s7'], label='s7 outgoing norm')
ax.plot(frame['epoch'], frame['output_weight_energy_long_fraction'], label='long output-weight energy fraction')
ax.set_xlabel('Epoch')
ax.set_ylabel('Output utilization')
ax.legend()
plt.show()

## Calibration
Use this table to verify that the non-baseline regularizers were calibrated from the exact shared epoch-5 fork point and that raw gradient scales are not degenerate.

In [ ]:
display(calibration.sort_values(['architecture', 'condition', 'seed']))